In [ ]:
import json
from pathlib import Path

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import confusion_matrix

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

ARTIFACTS_DIR = BASE_DIR / "artifacts"

In [ ]:
sklearn_model = joblib.load(ARTIFACTS_DIR / "sklearn_model.joblib")
scaler = joblib.load(ARTIFACTS_DIR / "scaler.joblib")

manual_weights = torch.load(ARTIFACTS_DIR / "manual_pytorch_weights.pt", weights_only=False)
w_manual, b_manual = manual_weights["w"], manual_weights["b"]

class LogisticRegressionModule(torch.nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = torch.nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x).squeeze(1)

standard_checkpoint = torch.load(ARTIFACTS_DIR / "standard_pytorch_model.pt", weights_only=False)
standard_model = LogisticRegressionModule(standard_checkpoint["n_features"])
standard_model.load_state_dict(standard_checkpoint["state_dict"])
standard_model.eval()

with open(ARTIFACTS_DIR / "feature_cols.json") as f:
    FEATURE_COLS = json.load(f)

test_df = pd.read_csv(ARTIFACTS_DIR / "test_data.csv", parse_dates=["Flight Date"])
comparison_df = pd.read_csv(ARTIFACTS_DIR / "comparison_metrics.csv", index_col=0)

In [ ]:
X_test = scaler.transform(test_df[FEATURE_COLS].values.astype(np.float64))
y_test = test_df["target"].values.astype(np.float64)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

with torch.no_grad():
    manual_probs = torch.sigmoid(X_test_t @ w_manual + b_manual).numpy()
    standard_probs = torch.sigmoid(standard_model(X_test_t)).numpy()
sklearn_probs = sklearn_model.predict_proba(X_test)[:, 1]

MODEL_PROBS = {
    "scikit-learn": sklearn_probs,
    "manual pytorch": manual_probs,
    "standard pytorch": standard_probs,
}

NUMERIC_FEATURES = [c for c in FEATURE_COLS if test_df[c].nunique() > 5]

# cost assumptions
COST_FN = 850
COST_FP = 15

In [ ]:
def show_comparison():
    fig, ax = plt.subplots()
    for name, probs in MODEL_PROBS.items():
        ax.hist(probs[y_test == 1], bins=20, alpha=0.5, label=name)
    ax.set_xlabel("predicted probability of Not Satisfied")
    ax.set_ylabel("count")
    ax.set_title("passengers who were actually Not Satisfied")
    ax.legend()
    table = comparison_df.reset_index().to_string(index=False)
    return fig, table

comparison_tab = gr.Interface(
    fn=show_comparison,
    inputs=None,
    outputs=[gr.Plot(label="prediction vs actual"), gr.Textbox(label="metrics", lines=6)],
    title="Model comparison",
)

In [ ]:
def show_feature(feature):
    fig, ax = plt.subplots()
    satisfied = test_df[test_df["target"] == 0][feature]
    not_satisfied = test_df[test_df["target"] == 1][feature]
    ax.hist(satisfied, bins=20, alpha=0.5, label="Satisfied")
    ax.hist(not_satisfied, bins=20, alpha=0.5, label="Not Satisfied")
    ax.set_xlabel(feature)
    ax.set_ylabel("count")
    ax.legend()
    return fig

distribution_tab = gr.Interface(
    fn=show_feature,
    inputs=gr.Dropdown(NUMERIC_FEATURES, value=NUMERIC_FEATURES[0], label="feature"),
    outputs=gr.Plot(),
    title="Feature distributions",
)

In [ ]:
def show_threshold(model_name, threshold, cost_fn, cost_fp):
    probs = MODEL_PROBS[model_name]
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()

    fig, ax = plt.subplots()
    cm = [[tn, fp], [fn, tp]]
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i][j], ha="center", va="center")
    ax.set_xticks([0, 1], labels=["pred Satisfied", "pred Not Satisfied"])
    ax.set_yticks([0, 1], labels=["actual Satisfied", "actual Not Satisfied"])
    ax.set_title(f"{model_name} @ threshold {threshold:.2f}")

    cost = fn * cost_fn + fp * cost_fp
    summary = (
        f"TP={tp}  FP={fp}  FN={fn}  TN={tn}\n"
        f"estimated business cost = {fn} missed x {cost_fn} + {fp} unneeded outreach x {cost_fp} = {cost:,.0f}"
    )
    return fig, summary

threshold_tab = gr.Interface(
    fn=show_threshold,
    inputs=[
        gr.Dropdown(list(MODEL_PROBS.keys()), value="scikit-learn", label="model"),
        gr.Slider(0, 1, value=0.3, step=0.01, label="threshold"),
        gr.Number(value=COST_FN, label="cost per missed at-risk customer"),
        gr.Number(value=COST_FP, label="cost per unnecessary outreach"),
    ],
    outputs=[gr.Plot(label="confusion matrix"), gr.Textbox(label="summary")],
    title="Threshold & business cost",
)

In [ ]:
demo = gr.TabbedInterface(
    [comparison_tab, distribution_tab, threshold_tab],
    ["Model comparison", "Feature distributions", "Threshold & cost"],
)

In [ ]:
demo.launch()